In [1]:
from google.colab import files

uploaded = files.upload()


Saving customer_incremental.csv to customer_incremental.csv


In [2]:
from google.colab import files

uploaded = files.upload()


Saving customer_master.csv to customer_master.csv


In [3]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

!pip install -q pyspark==4.0.0 delta-spark==4.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Week7 Delta Lake MERGE")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.0


# Q1. Load Dataset into a Delta Table

In [2]:
from pyspark.sql.types import *

master_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("customer_master.csv")

master_df.show(5)

+-----------+----------+---------+--------------------+--------------------+-------+-----------+-------+-----------+----------+--------+
|customer_id|first_name|last_name|               email|               phone|   city|      state|country|signup_date|membership|  status|
+-----------+----------+---------+--------------------+--------------------+-------+-----------+-------+-----------+----------+--------+
|  CUST00001|     Diana|     Bass|paulerica@example...|          9935730990| Mumbai|      Delhi|  India| 2023-10-17|  Platinum|Inactive|
|  CUST00002|  Patricia|     Pena|samanthahoward@ex...|          9489223748|   Pune|      Bihar|  India| 2022-09-08|  Platinum|  Active|
|  CUST00003|    Dalton|    Smith|carolyn94@example...| (913)361-4471x19728| Jaipur|Maharashtra|  India| 2023-11-24|  Platinum|Inactive|
|  CUST00004|      Rose|     Hall|justinadkins@exam...|   (329)594-8104x439|  Delhi|Maharashtra|  India| 2022-02-02|    Silver|  Active|
|  CUST00005|   Phyllis|     Cook|antonio

In [3]:
master_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- membership: string (nullable = true)
 |-- status: string (nullable = true)



In [4]:
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/customer_master_delta")

In [5]:
delta_master = spark.read \
    .format("delta") \
    .load("/content/customer_master_delta")

delta_master.show(5)

+-----------+----------+---------+--------------------+--------------------+-------+-----------+-------+-----------+----------+--------+
|customer_id|first_name|last_name|               email|               phone|   city|      state|country|signup_date|membership|  status|
+-----------+----------+---------+--------------------+--------------------+-------+-----------+-------+-----------+----------+--------+
|  CUST00001|     Diana|     Bass|paulerica@example...|          9935730990| Mumbai|      Delhi|  India| 2023-10-17|  Platinum|Inactive|
|  CUST00002|  Patricia|     Pena|samanthahoward@ex...|          9489223748|   Pune|      Bihar|  India| 2022-09-08|  Platinum|  Active|
|  CUST00003|    Dalton|    Smith|carolyn94@example...| (913)361-4471x19728| Jaipur|Maharashtra|  India| 2023-11-24|  Platinum|Inactive|
|  CUST00004|      Rose|     Hall|justinadkins@exam...|   (329)594-8104x439|  Delhi|Maharashtra|  India| 2022-02-02|    Silver|  Active|
|  CUST00005|   Phyllis|     Cook|antonio

# Q2. Handle Missing Values and Remove Duplicate Records

In [6]:
from pyspark.sql.functions import col, count, when

master_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in master_df.columns
]).show()

+-----------+----------+---------+-----+-----+----+-----+-------+-----------+----------+------+
|customer_id|first_name|last_name|email|phone|city|state|country|signup_date|membership|status|
+-----------+----------+---------+-----+-----+----+-----+-------+-----------+----------+------+
|          0|         0|        0|    0|   80|   0|    0|      0|          0|        50|     0|
+-----------+----------+---------+-----+-----+----+-----+-------+-----------+----------+------+



In [7]:
clean_df = master_df.dropna()

In [8]:
clean_df = clean_df.dropDuplicates()

In [9]:
print("Original Rows:", master_df.count())
print("Clean Rows:", clean_df.count())

Original Rows: 5030
Clean Rows: 4900


In [17]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/customer_master_delta")

# Q3. Load Incremental Dataset

In [18]:
incremental_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("customer_incremental.csv")

incremental_df.show(5)

+-----------+----------+---------+--------------------+--------------------+---------+-----------+-------+-----------+----------+--------+
|customer_id|first_name|last_name|               email|               phone|     city|      state|country|signup_date|membership|  status|
+-----------+----------+---------+--------------------+--------------------+---------+-----------+-------+-----------+----------+--------+
|  CUST02526|   Allison|    Jones|torresryan@exampl...|+1-935-467-4849x6...|    Delhi|      Bihar|  India| 2023-08-20|  Platinum|  Active|
|  CUST02923|     James|   Martin|cooperamanda@exam...|    887.364.6634x853|    Delhi|      Bihar|  India| 2024-01-11|    Silver|Inactive|
|  CUST00890|    Robert|     Wade| david74@example.net|    001-886-900-1466|    Patna|  Telangana|  India| 2024-04-16|  Platinum|  Active|
|  CUST00658|   Darlene|  Holland|beckdiane@example...|     +1-648-500-1309|Hyderabad|  Telangana|  India| 2023-02-26|  Platinum|  Active|
|  CUST04317|   Jeffrey| Mi

In [19]:
print("Incremental Rows:", incremental_df.count())

Incremental Rows: 500


In [20]:
incremental_df = incremental_df.dropDuplicates(["customer_id"])

In [21]:
print("Rows After Removing Duplicates:", incremental_df.count())

Rows After Removing Duplicates: 499


In [22]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/customer_master_delta")

# Q4. Apply MERGE Operation (Update Existing and Insert New Records)

In [23]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/content/customer_master_delta"
)

In [24]:
(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(
        set={
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "email": "source.email",
            "phone": "source.phone",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country",
            "signup_date": "source.signup_date",
            "membership": "source.membership",
            "status": "source.status"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "email": "source.email",
            "phone": "source.phone",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country",
            "signup_date": "source.signup_date",
            "membership": "source.membership",
            "status": "source.status"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

# Q5. Validate Results

In [25]:
final_df = spark.read \
    .format("delta") \
    .load("/content/customer_master_delta")

In [26]:
print("Final Row Count:", final_df.count())

Final Row Count: 5119


In [27]:
from pyspark.sql.functions import col

duplicate_count = (
    final_df.groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Customer IDs:", duplicate_count)

Duplicate Customer IDs: 0


In [28]:
print("Original Dataset :", master_df.count())
print("Clean Dataset :", clean_df.count())
print("Incremental Dataset :", incremental_df.count())
print("Final Dataset :", final_df.count())

Original Dataset : 5030
Clean Dataset : 4900
Incremental Dataset : 499
Final Dataset : 5119


# Q6. Display Final Dataset and Summary

In [29]:
final_df.show(20, truncate=False)

+-----------+----------+----------+---------------------------+----------------------+---------+-------------+-------+-----------+----------+--------+
|customer_id|first_name|last_name |email                      |phone                 |city     |state        |country|signup_date|membership|status  |
+-----------+----------+----------+---------------------------+----------------------+---------+-------------+-------+-----------+----------+--------+
|CUST00001  |Diana     |Bass      |paulerica@example.org      |9935730990            |Mumbai   |Delhi        |India  |2023-10-17 |Platinum  |Inactive|
|CUST00002  |Patricia  |Pena      |samanthahoward@example.net |9489223748            |Pune     |Bihar        |India  |2022-09-08 |Platinum  |Active  |
|CUST00003  |Dalton    |Smith     |carolyn94@example.com      |(913)361-4471x19728   |Jaipur   |Maharashtra  |India  |2023-11-24 |Platinum  |Inactive|
|CUST00004  |Rose      |Hall      |justinadkins@example.com   |(329)594-8104x439     |Delhi   

In [30]:
final_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- membership: string (nullable = true)
 |-- status: string (nullable = true)



# Assignment Summary

This assignment demonstrates incremental data processing using Delta Lake.

Tasks Completed:
- Loaded the customer dataset into a Delta table.
- Performed data cleaning by removing null values and duplicate records.
- Loaded an incremental customer dataset.
- Applied Delta Lake MERGE operation to update existing records and insert new records.
- Validated the final dataset by checking row counts and duplicate customer IDs.
- Displayed the final merged dataset and schema successfully.

# Conclusion

The objective of this assignment was to implement incremental data processing using Delta Lake.

The customer dataset was successfully loaded into a Delta table, cleaned by removing missing values and duplicate records, and updated using an incremental dataset through the Delta Lake MERGE operation. The final dataset was validated by checking the row count and ensuring that no duplicate customer IDs remained.

This assignment demonstrates how Delta Lake supports efficient upsert operations and reliable data management for incremental ETL pipelines.